# Day 7 · Exercise 1: Estimate Tokens and Check Fit

**What you'll build:** Two functions — `estimate_tokens(text: str) -> int` and `will_fit(text: str) -> bool` — that tell you whether a document is likely within the model's context window before you send it.

**Why it matters:** Sending a document that's too long either triggers an error or silently drops content. Checking first — with a cheap character-count estimate — means you can route long documents through a splitting step instead of discovering the problem at runtime.

> **No Ollama needed** for this exercise — it is pure Python.

## Your Implementation

In [ ]:
MODEL = "llama3.2"

# A conservative estimate for the local Ollama context window.
# 1 token ≈ 4 characters; reserve 200 tokens for the system prompt + response headroom.
CONTEXT_WINDOW_TOKENS  = 2048
CHARS_PER_TOKEN        = 4
PROMPT_OVERHEAD_TOKENS = 200

MAX_DOC_TOKENS = CONTEXT_WINDOW_TOKENS - PROMPT_OVERHEAD_TOKENS  # 1848 tokens
MAX_DOC_CHARS  = MAX_DOC_TOKENS * CHARS_PER_TOKEN                # 7392 characters


def estimate_tokens(text: str) -> int:
    """Return a rough token count for English prose.

    Uses the 1 token ≈ 4 characters approximation. This is a heuristic,
    not an exact count — good enough for a feasibility check before sending
    a document to the model.

    Args:
        text: The document string to measure.

    Returns:
        An integer token estimate (len(text) // 4).

    Example:
        >>> estimate_tokens("Hello world!")
        3
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────


def will_fit(text: str) -> bool:
    """Return True if the document is likely within the model's context window.

    Calls estimate_tokens() and compares the result to MAX_DOC_TOKENS.
    Returns True when the estimate is less than or equal to MAX_DOC_TOKENS,
    meaning the document is likely safe to send in one call.

    Args:
        text: The document string to check.

    Returns:
        True if estimate_tokens(text) <= MAX_DOC_TOKENS, else False.

    Example:
        >>> will_fit("Short sentence.")
        True
        >>> will_fit("A" * 10_000)
        False
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 5 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

# Sample documents for testing
_SHORT_DOC = "Quarterly revenue increased by 12 percent year-over-year."
_LONG_DOC  = "A" * 10_000   # 10 000 chars ≈ 2 500 tokens — well over the limit


def _run_checks():
    score, total = 0, 5

    # Check 1: both functions exist and are callable
    try:
        assert callable(estimate_tokens), 'estimate_tokens is not defined'
        assert callable(will_fit),        'will_fit is not defined'
        print(f'{_PASS} Check 1/{total}: estimate_tokens and will_fit are defined and callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: estimate_tokens returns an integer for a known input
    try:
        result = estimate_tokens("Hello world!")   # 12 chars → 12 // 4 = 3
        assert isinstance(result, int), f'expected int, got {type(result).__name__}'
        assert result == 3, f'estimate_tokens("Hello world!") should be 3, got {result}'
        print(f'{_PASS} Check 2/{total}: estimate_tokens("Hello world!") == 3')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')

    # Check 3: short document fits
    try:
        result = will_fit(_SHORT_DOC)
        assert result is True, f'expected True for short doc, got {result!r}'
        print(f'{_PASS} Check 3/{total}: short document ({len(_SHORT_DOC)} chars) → will_fit returns True')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: long document does not fit
    try:
        result = will_fit(_LONG_DOC)
        assert result is False, f'expected False for 10 000-char doc, got {result!r}'
        print(f'{_PASS} Check 4/{total}: long document ({len(_LONG_DOC)} chars) → will_fit returns False')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    # Check 5: boundary — exactly MAX_DOC_CHARS fits; MAX_DOC_CHARS + 4 does not
    # (integer division means the token estimate only increments every 4 characters)
    try:
        at_boundary = will_fit("A" * MAX_DOC_CHARS)        # 7392 chars → 1848 tokens ≤ 1848 → True
        just_over   = will_fit("A" * (MAX_DOC_CHARS + 4))  # 7396 chars → 1849 tokens > 1848 → False
        assert at_boundary is True,  (
            f'expected True at MAX_DOC_CHARS ({MAX_DOC_CHARS}), got {at_boundary!r}'
        )
        assert just_over   is False, (
            f'expected False at MAX_DOC_CHARS + 4 ({MAX_DOC_CHARS + 4}), got {just_over!r}'
        )
        print(
            f'{_PASS} Check 5/{total}: boundary — '
            f'"A"×{MAX_DOC_CHARS} → True; "A"×{MAX_DOC_CHARS + 4} → False'
        )
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 5/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {score}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

The token estimate uses integer division, which means `will_fit` returns `True`
for any string up to `MAX_DOC_CHARS + 3` characters (they all divide to 1848
tokens). A stricter check would round *up* instead.

Try implementing `estimate_tokens_ceil(text: str) -> int` using `math.ceil`:

```python
import math

def estimate_tokens_ceil(text: str) -> int:
    return math.ceil(len(text) / CHARS_PER_TOKEN)

def will_fit_strict(text: str) -> bool:
    return estimate_tokens_ceil(text) <= MAX_DOC_TOKENS
```

Compare both functions at `MAX_DOC_CHARS + 1`:

```python
print(estimate_tokens("A" * (MAX_DOC_CHARS + 1)))       # 1848 — fits with floor
print(estimate_tokens_ceil("A" * (MAX_DOC_CHARS + 1)))  # 1849 — does not fit with ceil
```

Which is more conservative? Which would you use in a production pipeline?

This foreshadows Lesson 2, where knowing exact chunk counts before splitting matters.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
def estimate_tokens(text: str) -> int:
    return len(text) // CHARS_PER_TOKEN


def will_fit(text: str) -> bool:
    return estimate_tokens(text) <= MAX_DOC_TOKENS
```

**Why this works:** `len(text) // 4` is integer division — it drops the remainder, so the estimate is always equal to or *lower* than the exact token count. That is intentional: if the estimate says a document fits, you send it. The comparison `<= MAX_DOC_TOKENS` (not `<`) is important: a document whose token estimate is exactly 1 848 is safe to send, so it should return `True`. One last detail: because `//` rounds down every four characters, a string of length `MAX_DOC_CHARS + 1` (7 393) still estimates to 1 848 tokens and returns `True`. The boundary only flips at `MAX_DOC_CHARS + 4` (7 396), where integer division first rounds up to 1 849.
</details>